In [1]:
import os
import getpass
from dotenv import load_dotenv
from groq import Groq

# Try loading from .env first
load_dotenv()

# If not found, ask user to enter it securely
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Initialize Groq client
client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Groq Client Initialized Successfully")

Groq Client Initialized Successfully


In [2]:
models = client.models.list()
for m in models.data:
    print(m.id)

qwen/qwen3-32b
openai/gpt-oss-20b
groq/compound-mini
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
moonshotai/kimi-k2-instruct-0905
llama-3.1-8b-instant
allam-2-7b
meta-llama/llama-4-maverick-17b-128e-instruct
llama-3.3-70b-versatile
canopylabs/orpheus-v1-english
meta-llama/llama-4-scout-17b-16e-instruct
moonshotai/kimi-k2-instruct
groq/compound
whisper-large-v3
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-guard-4-12b
openai/gpt-oss-120b
whisper-large-v3-turbo


In [3]:
# Switched to Llama since Groq migrated away from Mixtral 8x7B
MODEL_NAME = "llama-3.1-8b-instant"

In [4]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
You are a Senior Software Engineer.
Be precise, technical, and solution-oriented.
Provide code snippets when necessary.
Explain errors clearly and suggest debugging steps.
"""
    },
    "billing": {
        "system_prompt": """
You are a Customer Billing Specialist.
Be empathetic and professional.
Explain refund policies clearly.
Guide the user through next steps politely.
"""
    },
    "general": {
        "system_prompt": """
You are a friendly general customer support assistant.
Answer casually and clearly.
If unsure, politely ask for clarification.
"""
    }
}

In [5]:
def route_prompt(user_input: str) -> str:
    """
    Classifies user input into one of:
    technical, billing, general
    
    Returns ONLY the category name.
    """
    
    routing_prompt = f"""
Classify the following text into one of these categories:
[technical, billing, general]

Return ONLY the category name.

Text:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,  # deterministic routing
        messages=[
            {"role": "system", "content": "You are an intent classifier."},
            {"role": "user", "content": routing_prompt}
        ]
    )
    
    category = response.choices[0].message.content.strip().lower()
    
    # safety fallback
    if category not in MODEL_CONFIG:
        return "general"
    
    return category

In [6]:
def process_request(user_input: str) -> str:
    """
    Full MoE flow:
    1. Route intent
    2. Select expert
    3. Generate response
    """
    
    category = route_prompt(user_input)
    system_prompt = MODEL_CONFIG[category]["system_prompt"]
    
    print(f"Routed to: {category.upper()} expert\n")
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.7,  # creative expert
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )
    
    return response.choices[0].message.content

In [7]:
print(process_request("My Python script is throwing an IndexError on line 5."))

Routed to: TECHNICAL expert

An `IndexError` typically occurs when you're trying to access an element in a list or other sequence that doesn't exist, or when the index you're trying to access is out of range.

To help you solve the issue, I'll need more information about your Python script. Please provide:

1. The specific error message that's being thrown.
2. The code snippet that's causing the error.
3. The relevant parts of your code (e.g., variables, function calls) around line 5.

That being said, here are some general steps you can take to troubleshoot the issue:

**Common causes of IndexError:**

1. **Out-of-range index**: Ensure that the index you're trying to access is within the bounds of the list or sequence.
2. **Missing or undefined list**: Check that the list or sequence you're trying to access is defined and not empty.
3. **Incorrect index type**: Make sure the index you're using is of the correct type (e.g., integer).

**Debugging steps:**

1. **Print the list or sequen

In [8]:
print(process_request("I was charged twice for my subscription this month."))

Routed to: BILLING expert

I'm so sorry to hear that you were charged twice for your subscription this month. I understand how frustrating this must be for you.

Don't worry, I'm here to help you resolve the issue. Our refund policy is as follows:

If you've been charged twice for your subscription, we'll process a refund for the duplicate charge, minus any applicable fees. This may take 3-5 business days to process, depending on your bank's processing time.

To proceed with the refund, could you please provide me with your account information and the date of the duplicate charge? This will help me locate the issue quickly and efficiently.

Additionally, if you'd like to avoid being charged again for the current month, I can temporarily suspend your subscription until the issue is resolved. Would you like me to do that for you?


In [9]:
print(process_request("Hi there! What services do you offer?"))

Routed to: GENERAL expert

Hello! We're glad you reached out to us. Our company offers a wide range of services, so I'd be happy to give you a rundown.

We have an e-commerce platform where you can buy and sell various products online, including electronics, clothing, home goods, and more. We also provide a marketplace for local service providers to offer their services, such as house cleaning, pet sitting, and handyman services.

Additionally, we have a digital storage service that allows you to store and manage your files securely online. You can access your files from anywhere, share them with others, and even collaborate on projects.

If you're interested in learning more about any of these services or have specific questions, feel free to ask and I'll do my best to help!


In [10]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
You are a Senior Software Engineer.
Be precise and technical.
Provide code fixes and debugging steps.
"""
    },
    "billing": {
        "system_prompt": """
You are a Billing Support Specialist.
Be empathetic and explain refund policies clearly.
Guide users professionally.
"""
    },
    "general": {
        "system_prompt": """
You are a friendly customer support assistant.
Answer clearly and politely.
"""
    },
    "tool": {
        "system_prompt": """
You are a tool router. If the system calls you,
it means a function should be executed instead of answering normally.
"""
    }
}

In [11]:
def route_prompt(user_input: str) -> str:
    routing_prompt = f"""
You are a strict intent classifier.

Classify the user message into ONE of these categories:
technical
billing
general
tool

Rules:
- If the user asks for real-time data (price, weather, stock, crypto), choose: tool
- If asking about code, bugs, errors → technical
- If asking about payment, charges, refunds → billing
- Otherwise → general

Respond with ONLY one word.

Message:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are an intent classifier."},
            {"role": "user", "content": routing_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()

    if category not in MODEL_CONFIG:
        return "general"

    return category

In [12]:
def get_bitcoin_price():
    return {
        "asset": "Bitcoin",
        "price_usd": 52340,
        "source": "Mock Market API"
    }

In [13]:
def process_request(user_input: str) -> str:
    category = route_prompt(user_input)

    print(f"Routed to: {category.upper()}")

    # ---- TOOL HANDLING ----
    if category == "tool":
        tool_result = get_bitcoin_price()
        
        return (
            f"Live Market Data\n"
            f"Asset: {tool_result['asset']}\n"
            f"Price (USD): ${tool_result['price_usd']}\n"
            f"Source: {tool_result['source']}"
        )

    # ---- LLM EXPERT HANDLING ----
    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

In [14]:

print(process_request("My Python script throws IndexError on line 5"))

Routed to: TECHNICAL
To solve this issue, we'll need more information about the script and the error message. However, I can provide a general approach to debug an `IndexError` in Python.

**Step 1: Examine the Error Message**

The error message should indicate the line number where the `IndexError` occurs. In this case, it's line 5. The message might look something like this:

```
IndexError: list index out of range
```

or

```python
IndexError: tuple index out of range
```

**Step 2: Review the Code**

Look at the code around line 5 and check the following:

* Are you trying to access an element in a list or tuple that is out of its valid index range?
* Are you trying to access a dictionary key that doesn't exist?

**Step 3: Investigate the Data**

Print or log the data that is being accessed around line 5 to see if it's what you expect. This can help you identify if the issue is related to the data or the code.

**Step 4: Simplify the Code**

Try to simplify the code around line 5 

In [15]:
print(process_request("I was charged twice for my subscription"))

Routed to: BILLING
I'm so sorry to hear that you've been charged twice for your subscription. I can imagine how frustrating that must be for you.

To help resolve this issue as quickly as possible, I'd like to explain our refund policy and the steps we'll take to rectify the situation.

Our refund policy states that if you're charged incorrectly, we'll refund the duplicate charge as soon as possible. In this case, since you've been charged twice for your subscription, we'll process a refund for the extra amount charged.

To initiate the refund process, I'll need to verify some information from you:

1. Can you please provide me with your order number or subscription ID so I can locate your account?
2. Can you confirm the amount that was charged twice?
3. Have you already received any refunds or notifications regarding this issue?

Once I have this information, I'll process the refund immediately. You can expect to see the refund credited back to your original payment method within 3-5 

In [16]:

print(process_request("What is the current price of Bitcoin?"))

Routed to: TOOL
Live Market Data
Asset: Bitcoin
Price (USD): $52340
Source: Mock Market API
